# HR Onboarding Support Agent: Data Pipeline and EDA

**AAI-510: Agentic AI Systems**  
**Final Team Project**  
**Data Engineer:** Emmi Bishop  
**Team Members:** Emmi Bishop, Peng Wang, Glen Salazar  

## Notebook Purpose

This notebook contains the Data Engineer portion of the final project. It loads the Employee Onboarding Effectiveness dataset, performs exploratory data analysis, cleans and validates the data, creates agent-ready features, and saves cleaned tables for the HR Onboarding Support Agent.

The cleaned outputs from this notebook will support agent tools for employee risk lookup, department analysis, location analysis, and onboarding recommendation generation.

## 1. Load Dataset

This section imports the Python libraries needed for data processing and loads the Employee Onboarding Effectiveness CSV file from the Databricks volume. The dataset is stored in Unity Catalog so it can be reused by the rest of the project.

In [0]:
# Import libraries for data loading, cleaning, and analysis

import pandas as pd
import numpy as np

# Make tables easier to view
pd.set_option("display.max_columns", None)

# File path from Databricks volume
file_path = "/Volumes/main/default/hr_onboarding_data/onboarding_dataset.csv"

# Load dataset
df = pd.read_csv(file_path)

# Preview the dataset
df.head()

,employee_id,department,location,hire_date,onboarding_survey_date,role_clarity_score,training_quality_score,manager_support_score,tooling_access_score,onboarding_nps,left_during_probation
0,E10070,Engineering,Remote-ES,2024-03-12,30/04/2024,9,4,5.0,9,6.8,0
1,E10218,Customer Success,Valencia,2025/06/01,08-04-2025,3,6,8.0,9,6.5,0
2,E10385,Engineering,Remote-ES,10-25-2024,2024/12/16,3,4,6.0,8,5.2,1
3,E10033,Engineering,Madrid,2025/01/04,2025/02/17,6,8,7.0,6,6.8,0
4,E10042,Data,Barcelona,02-01-2025,03-21-2025,9,9,9.0,8,8.8,0


## 2. Initial Data Inspection

This section checks the basic structure of the dataset, including the number of rows and columns, column names, data types, missing values, and duplicate employee IDs. These checks help identify what cleaning is needed before the data can be used by the agent.

In [0]:
# Initial dataset inspection

print("Dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values by column:")
print(df.isnull().sum())

print("\nNumber of duplicate employee IDs:")
print(df["employee_id"].duplicated().sum())

Dataset shape:
(405, 11)

Column names:
['employee_id', 'department', 'location', 'hire_date', 'onboarding_survey_date', 'role_clarity_score', 'training_quality_score', 'manager_support_score', 'tooling_access_score', 'onboarding_nps', 'left_during_probation']

Data types:
employee_id                object
department                 object
location                   object
hire_date                  object
onboarding_survey_date     object
role_clarity_score          int64
training_quality_score      int64
manager_support_score     float64
tooling_access_score        int64
onboarding_nps            float64
left_during_probation       int64
dtype: object

Missing values by column:
employee_id                0
department                 0
location                   0
hire_date                  0
onboarding_survey_date     0
role_clarity_score         0
training_quality_score     0
manager_support_score     10
tooling_access_score       0
onboarding_nps            10
left_during_probation

In [0]:
# Basic EDA summaries

print("Unique departments:")
print(df["department"].value_counts())

print("\nUnique locations:")
print(df["location"].value_counts())

print("\nProbation outcome counts:")
print(df["left_during_probation"].value_counts())

print("\nProbation outcome percentage:")
print(df["left_during_probation"].value_counts(normalize=True) * 100)

Unique departments:
department
Product             62
Sales               60
Finance             57
Data                49
Marketing           48
Customer Success    46
Engineering         44
HR                  39
Name: count, dtype: int64

Unique locations:
location
Madrid       95
Remote-ES    80
Barcelona    80
Seville      76
Valencia     74
Name: count, dtype: int64

Probation outcome counts:
left_during_probation
0    316
1     89
Name: count, dtype: int64

Probation outcome percentage:
left_during_probation
0    78.024691
1    21.975309
Name: proportion, dtype: float64


The dataset contains 405 rows and 11 columns. The initial inspection shows that the date columns are stored as text/object fields and need to be converted to datetime format. The dataset also contains missing values in `manager_support_score` and `onboarding_nps`, which will be handled during cleaning.

## 3. Score Validation and Missing Values

This section validates the onboarding survey score columns and checks for missing values. These checks are important because the agent will use these scores to identify onboarding trends, summarize risk, and generate HR recommendations.

In [0]:
# Validate score columns and check missing values

score_cols = [
    "role_clarity_score",
    "training_quality_score",
    "manager_support_score",
    "tooling_access_score",
    "onboarding_nps"
]

print("Missing values by column:")
print(df.isnull().sum())

print("\nScore column summary:")
print(df[score_cols].describe())

print("\nMinimum and maximum values for score columns:")
print(df[score_cols].agg(["min", "max"]))

print("\nRows with scores outside expected 1-10 range:")
for col in score_cols:
    invalid_rows = df[(df[col] < 1) | (df[col] > 10)]
    print(f"{col}: {len(invalid_rows)} invalid rows")

Missing values by column:
employee_id                0
department                 0
location                   0
hire_date                  0
onboarding_survey_date     0
role_clarity_score         0
training_quality_score     0
manager_support_score     10
tooling_access_score       0
onboarding_nps            10
left_during_probation      0
dtype: int64

Score column summary:
       role_clarity_score  training_quality_score  manager_support_score  \
count          405.000000              405.000000             395.000000   
mean             5.970370                6.451852               6.187342   
std              2.014578                1.671182               1.913177   
min              3.000000                4.000000               3.000000   
25%              4.000000                5.000000               5.000000   
50%              6.000000                7.000000               6.000000   
75%              8.000000                8.000000               8.000000   
max        

The score validation showed that most survey score fields are complete, but `manager_support_score` and `onboarding_nps` each    contain 10 missing values. These missing values will be handled during cleaning using median imputation so the records can remain in the dataset without heavily skewing the analysis.

## 4. Data Cleaning

This section creates a cleaned copy of the dataset. The cleaning process converts the mixed date formats in `hire_date` and `onboarding_survey_date` into datetime fields. It also fills missing values in `manager_support_score` and `onboarding_nps` using median imputation.

In [0]:
# Create a cleaned copy of the dataset

df_clean = df.copy()

# Function to handle mixed date formats
def parse_mixed_dates(date_series):
    # Try pandas mixed-format parsing first
    parsed_dates = pd.to_datetime(date_series, errors="coerce", format="mixed")
    
    # Try additional common formats for anything still missing
    date_formats = [
        "%Y-%m-%d",
        "%m-%d-%Y",
        "%d/%m/%Y",
        "%Y/%m/%d",
        "%m/%d/%Y",
        "%d-%m-%Y"
    ]
    
    for date_format in date_formats:
        missing_mask = parsed_dates.isna()
        parsed_dates.loc[missing_mask] = pd.to_datetime(
            date_series.loc[missing_mask],
            errors="coerce",
            format=date_format
        )
    
    return parsed_dates

# Convert date columns to datetime format
df_clean["hire_date"] = parse_mixed_dates(df_clean["hire_date"])
df_clean["onboarding_survey_date"] = parse_mixed_dates(df_clean["onboarding_survey_date"])

# Fill missing numeric values with the column median
df_clean["manager_support_score"] = df_clean["manager_support_score"].fillna(
    df_clean["manager_support_score"].median()
)

df_clean["onboarding_nps"] = df_clean["onboarding_nps"].fillna(
    df_clean["onboarding_nps"].median()
)

# Confirm cleaning results
print("Missing values after cleaning:")
print(df_clean.isnull().sum())

print("\nCleaned data types:")
print(df_clean.dtypes)

display(df_clean.head())

Missing values after cleaning:
employee_id               0
department                0
location                  0
hire_date                 0
onboarding_survey_date    0
role_clarity_score        0
training_quality_score    0
manager_support_score     0
tooling_access_score      0
onboarding_nps            0
left_during_probation     0
dtype: int64

Cleaned data types:
employee_id                       object
department                        object
location                          object
hire_date                 datetime64[ns]
onboarding_survey_date    datetime64[ns]
role_clarity_score                 int64
training_quality_score             int64
manager_support_score            float64
tooling_access_score               int64
onboarding_nps                   float64
left_during_probation              int64
dtype: object


employee_id,department,location,hire_date,onboarding_survey_date,role_clarity_score,training_quality_score,manager_support_score,tooling_access_score,onboarding_nps,left_during_probation
E10070,Engineering,Remote-ES,2024-03-12T00:00:00.000Z,2024-04-30T00:00:00.000Z,9,4,5.0,9,6.8,0
E10218,Customer Success,Valencia,2025-06-01T00:00:00.000Z,2025-08-04T00:00:00.000Z,3,6,8.0,9,6.5,0
E10385,Engineering,Remote-ES,2024-10-25T00:00:00.000Z,2024-12-16T00:00:00.000Z,3,4,6.0,8,5.2,1
E10033,Engineering,Madrid,2025-01-04T00:00:00.000Z,2025-02-17T00:00:00.000Z,6,8,7.0,6,6.8,0
E10042,Data,Barcelona,2025-02-01T00:00:00.000Z,2025-03-21T00:00:00.000Z,9,9,9.0,8,8.8,0


After cleaning, the dataset has no missing values in the main columns. The date fields were successfully converted into datetime format, and the missing values in `manager_support_score` and `onboarding_nps` were filled using the median of each column. This prepares the dataset for feature engineering and agent-ready analysis.

## 5. Feature Engineering

This section creates new fields that make the dataset more useful for the agent. The pipeline adds an anonymized employee key, calculates the number of days between hire date and onboarding survey date, creates an overall onboarding score, counts low-score indicators, and assigns each employee an onboarding risk category.

In [0]:
# Feature engineering for agent-ready dataset

# Create anonymized employee key for safer downstream use
df_clean["employee_key"] = ["EMP_" + str(i + 1).zfill(4) for i in range(len(df_clean))]

# Calculate number of days between hire date and onboarding survey date
df_clean["days_to_survey"] = (
    df_clean["onboarding_survey_date"] - df_clean["hire_date"]
).dt.days

# Create an overall onboarding score using the main survey score columns
survey_score_cols = [
    "role_clarity_score",
    "training_quality_score",
    "manager_support_score",
    "tooling_access_score",
    "onboarding_nps"
]

df_clean["overall_onboarding_score"] = df_clean[survey_score_cols].mean(axis=1).round(2)

# Count how many onboarding metrics are low for each employee
df_clean["low_score_count"] = (df_clean[survey_score_cols] <= 4).sum(axis=1)

# Create onboarding risk category
conditions = [
    (df_clean["left_during_probation"] == 1) | 
    (df_clean["overall_onboarding_score"] <= 4.5) | 
    (df_clean["low_score_count"] >= 2),
    
    (df_clean["overall_onboarding_score"] <= 6.5) | 
    (df_clean["low_score_count"] == 1)
]

choices = ["High Risk", "Medium Risk"]

df_clean["onboarding_risk_category"] = np.select(
    conditions,
    choices,
    default="Low Risk"
)

# Check new feature results
print("Risk category counts:")
print(df_clean["onboarding_risk_category"].value_counts())

print("\nDays to survey summary:")
print(df_clean["days_to_survey"].describe())

print("\nRows where survey date is before hire date:")
print((df_clean["days_to_survey"] < 0).sum())

display(df_clean.head())

Risk category counts:
onboarding_risk_category
Medium Risk    170
Low Risk       120
High Risk      115
Name: count, dtype: int64

Days to survey summary:
count    405.000000
mean      64.822222
std       67.223235
min     -271.000000
25%       43.000000
50%       60.000000
75%       77.000000
max      346.000000
Name: days_to_survey, dtype: float64

Rows where survey date is before hire date:
21


employee_id,department,location,hire_date,onboarding_survey_date,role_clarity_score,training_quality_score,manager_support_score,tooling_access_score,onboarding_nps,left_during_probation,employee_key,days_to_survey,overall_onboarding_score,low_score_count,onboarding_risk_category
E10070,Engineering,Remote-ES,2024-03-12T00:00:00.000Z,2024-04-30T00:00:00.000Z,9,4,5.0,9,6.8,0,EMP_0001,49,6.76,1,Medium Risk
E10218,Customer Success,Valencia,2025-06-01T00:00:00.000Z,2025-08-04T00:00:00.000Z,3,6,8.0,9,6.5,0,EMP_0002,64,6.5,1,Medium Risk
E10385,Engineering,Remote-ES,2024-10-25T00:00:00.000Z,2024-12-16T00:00:00.000Z,3,4,6.0,8,5.2,1,EMP_0003,52,5.24,2,High Risk
E10033,Engineering,Madrid,2025-01-04T00:00:00.000Z,2025-02-17T00:00:00.000Z,6,8,7.0,6,6.8,0,EMP_0004,44,6.76,0,Low Risk
E10042,Data,Barcelona,2025-02-01T00:00:00.000Z,2025-03-21T00:00:00.000Z,9,9,9.0,8,8.8,0,EMP_0005,48,8.76,0,Low Risk


The engineered features create an agent-ready version of the dataset. The `onboarding_risk_category` field can support tools that identify employees who may need additional onboarding support.

This step also revealed a data quality issue: 21 records have an onboarding survey date before the hire date. These rows should be flagged and handled before using `days_to_survey` for analysis.

## 6. Date Quality Check

This section checks whether any onboarding survey dates occur before the employee hire date. This is a data quality issue because onboarding surveys should normally happen after the employee starts.

In [0]:
# Inspect rows where onboarding survey date is before hire date

negative_survey_rows = df_clean[df_clean["days_to_survey"] < 0][
    [
        "employee_id",
        "employee_key",
        "department",
        "location",
        "hire_date",
        "onboarding_survey_date",
        "days_to_survey",
        "overall_onboarding_score",
        "onboarding_risk_category"
    ]
]

print("Number of rows where survey date is before hire date:")
print(len(negative_survey_rows))

display(negative_survey_rows)

Number of rows where survey date is before hire date:
21


employee_id,employee_key,department,location,hire_date,onboarding_survey_date,days_to_survey,overall_onboarding_score,onboarding_risk_category
E10332,EMP_0008,Product,Madrid,2025-05-09T00:00:00.000Z,2025-01-07T00:00:00.000Z,-122,7.0,Low Risk
E10232,EMP_0024,Product,Remote-ES,2024-09-10T00:00:00.000Z,2024-03-12T00:00:00.000Z,-182,6.76,Medium Risk
E10113,EMP_0105,Finance,Seville,2026-09-01T00:00:00.000Z,2026-03-23T00:00:00.000Z,-162,5.24,Medium Risk
E10299,EMP_0114,Data,Valencia,2024-09-24T00:00:00.000Z,2024-04-11T00:00:00.000Z,-166,6.5,Medium Risk
E10301,EMP_0118,Customer Success,Seville,2025-05-27T00:00:00.000Z,2025-04-08T00:00:00.000Z,-49,5.24,High Risk
E10165,EMP_0133,Customer Success,Valencia,2025-07-19T00:00:00.000Z,2025-05-10T00:00:00.000Z,-70,7.0,Medium Risk
E10357,EMP_0150,Sales,Madrid,2024-03-30T00:00:00.000Z,2024-03-06T00:00:00.000Z,-24,6.24,Medium Risk
E10010,EMP_0156,Sales,Valencia,2025-06-09T00:00:00.000Z,2025-05-08T00:00:00.000Z,-32,5.0,High Risk
E10122,EMP_0184,Finance,Madrid,2024-11-08T00:00:00.000Z,2024-09-27T00:00:00.000Z,-42,4.5,High Risk
E10067,EMP_0187,Marketing,Remote-ES,2026-11-02T00:00:00.000Z,2026-02-04T00:00:00.000Z,-271,6.24,Medium Risk


The validation found 21 records where the onboarding survey date is earlier than the hire date. These rows were kept because the onboarding scores are still useful for analysis, but they need to be flagged so the invalid `days_to_survey` values do not affect timing-based analysis.

## 7. Handle Date Issues

This section handles the records where the onboarding survey date occurs before the hire date. Instead of deleting these rows, the pipeline flags them with `date_issue_flag` and sets their invalid `days_to_survey` values to missing.

In [0]:
# Handle date quality issue where survey date is before hire date

# Create a flag for rows with date issues
df_clean["date_issue_flag"] = df_clean["days_to_survey"] < 0

# Set invalid days_to_survey values to missing
df_clean.loc[df_clean["date_issue_flag"], "days_to_survey"] = np.nan

# Check results
print("Rows with date issue flag:")
print(df_clean["date_issue_flag"].sum())

print("\nUpdated days_to_survey summary:")
print(df_clean["days_to_survey"].describe())

print("\nMissing values after date issue handling:")
print(df_clean.isnull().sum())

Rows with date issue flag:
21

Updated days_to_survey summary:
count    384.000000
mean      73.315104
std       55.624320
min        8.000000
25%       46.750000
50%       62.000000
75%       78.250000
max      346.000000
Name: days_to_survey, dtype: float64

Missing values after date issue handling:
employee_id                  0
department                   0
location                     0
hire_date                    0
onboarding_survey_date       0
role_clarity_score           0
training_quality_score       0
manager_support_score        0
tooling_access_score         0
onboarding_nps               0
left_during_probation        0
employee_key                 0
days_to_survey              21
overall_onboarding_score     0
low_score_count              0
onboarding_risk_category     0
date_issue_flag              0
dtype: int64


After handling the date issue, 21 records are flagged with `date_issue_flag`. The invalid negative `days_to_survey` values were replaced with missing values, leaving 384 valid records for timing-based analysis. This keeps the survey score data available while preventing incorrect date values from affecting the agent’s analysis.

## 8. Agent-Ready Summary Tables

This section creates summary tables by department, location, and onboarding risk category. These tables are designed to support the agent’s analytical tools and make common HR questions easier to answer.

In [0]:
# Create summary tables for agent tools

# Department-level summary
department_summary = df_clean.groupby("department").agg(
    employee_count=("employee_key", "count"),
    avg_role_clarity=("role_clarity_score", "mean"),
    avg_training_quality=("training_quality_score", "mean"),
    avg_manager_support=("manager_support_score", "mean"),
    avg_tooling_access=("tooling_access_score", "mean"),
    avg_onboarding_nps=("onboarding_nps", "mean"),
    avg_overall_onboarding_score=("overall_onboarding_score", "mean"),
    probation_loss_rate=("left_during_probation", "mean"),
    high_risk_count=("onboarding_risk_category", lambda x: (x == "High Risk").sum())
).round(2).reset_index()

# Location-level summary
location_summary = df_clean.groupby("location").agg(
    employee_count=("employee_key", "count"),
    avg_role_clarity=("role_clarity_score", "mean"),
    avg_training_quality=("training_quality_score", "mean"),
    avg_manager_support=("manager_support_score", "mean"),
    avg_tooling_access=("tooling_access_score", "mean"),
    avg_onboarding_nps=("onboarding_nps", "mean"),
    avg_overall_onboarding_score=("overall_onboarding_score", "mean"),
    probation_loss_rate=("left_during_probation", "mean"),
    high_risk_count=("onboarding_risk_category", lambda x: (x == "High Risk").sum())
).round(2).reset_index()

# Risk category summary
risk_summary = df_clean.groupby("onboarding_risk_category").agg(
    employee_count=("employee_key", "count"),
    avg_overall_onboarding_score=("overall_onboarding_score", "mean"),
    avg_manager_support=("manager_support_score", "mean"),
    avg_training_quality=("training_quality_score", "mean"),
    probation_loss_rate=("left_during_probation", "mean")
).round(2).reset_index()

print("Department Summary:")
display(department_summary)

print("Location Summary:")
display(location_summary)

print("Risk Summary:")
display(risk_summary)

Department Summary:


department,employee_count,avg_role_clarity,avg_training_quality,avg_manager_support,avg_tooling_access,avg_onboarding_nps,avg_overall_onboarding_score,probation_loss_rate,high_risk_count
Customer Success,46,6.33,6.43,6.41,7.09,6.56,6.56,0.15,11
Data,49,5.71,6.67,6.04,7.2,6.41,6.41,0.14,9
Engineering,44,5.77,5.98,6.43,7.11,6.36,6.33,0.3,15
Finance,57,6.16,6.49,6.04,6.79,6.35,6.36,0.16,14
HR,39,6.13,6.46,6.1,6.72,6.37,6.36,0.26,14
Marketing,48,5.65,6.6,6.5,7.21,6.45,6.48,0.12,9
Product,62,6.05,6.31,5.98,7.11,6.37,6.36,0.24,18
Sales,60,5.95,6.62,6.08,6.95,6.42,6.4,0.37,25


Location Summary:


location,employee_count,avg_role_clarity,avg_training_quality,avg_manager_support,avg_tooling_access,avg_onboarding_nps,avg_overall_onboarding_score,probation_loss_rate,high_risk_count
Barcelona,80,5.94,6.41,6.24,6.91,6.37,6.37,0.18,19
Madrid,95,6.02,6.69,6.26,6.98,6.51,6.49,0.2,24
Remote-ES,80,5.98,6.34,6.18,7.05,6.39,6.39,0.2,25
Seville,76,5.7,6.38,6.04,6.95,6.25,6.26,0.29,27
Valencia,74,6.22,6.38,6.18,7.26,6.5,6.51,0.24,20


Risk Summary:


onboarding_risk_category,employee_count,avg_overall_onboarding_score,avg_manager_support,avg_training_quality,probation_loss_rate
High Risk,115,5.7,5.08,5.63,0.77
Low Risk,120,7.3,7.25,7.12,0.0
Medium Risk,170,6.25,6.18,6.54,0.0


The summary tables provide structured outputs that the AI Engineer can connect to agent tools. For example, the agent can use these tables to answer questions about departments with lower onboarding scores, locations with higher probation loss rates, or groups with higher onboarding risk.

## 9. Save Processed Outputs

This section saves the cleaned agent-ready dataset and summary tables as CSV files in the Databricks volume. The original `employee_id` field is removed from the agent-ready dataset and replaced with the anonymized `employee_key` for safer downstream use.

In [0]:
# Save cleaned outputs for agent use

import os

# Output folder in Databricks volume
output_dir = "/Volumes/main/default/hr_onboarding_data/processed_outputs"
os.makedirs(output_dir, exist_ok=True)

# Create an agent-ready version without the original employee_id
# This keeps the anonymized employee_key instead
agent_ready_df = df_clean.drop(columns=["employee_id"])

# Save cleaned datasets and summary tables
agent_ready_df.to_csv(f"{output_dir}/cleaned_onboarding_agent_ready.csv", index=False)
department_summary.to_csv(f"{output_dir}/department_summary.csv", index=False)
location_summary.to_csv(f"{output_dir}/location_summary.csv", index=False)
risk_summary.to_csv(f"{output_dir}/risk_summary.csv", index=False)

print("Saved processed outputs to:")
print(output_dir)

print("\nFiles saved:")
print("- cleaned_onboarding_agent_ready.csv")
print("- department_summary.csv")
print("- location_summary.csv")
print("- risk_summary.csv")

Saved processed outputs to:
/Volumes/main/default/hr_onboarding_data/processed_outputs

Files saved:
- cleaned_onboarding_agent_ready.csv
- department_summary.csv
- location_summary.csv
- risk_summary.csv


The processed CSV files were saved successfully to the project volume. These files include the cleaned agent-ready dataset, department summary, location summary, and risk summary. These outputs can be shared with the AI Engineer for tool development and agent testing.

## 10. Create Queryable Databricks Tables

This section converts the cleaned pandas DataFrames into Spark DataFrames and saves them as Databricks tables. Creating queryable tables makes it easier for the agent to access the cleaned dataset and summary outputs through SQL-based tools.

In [0]:
# Create queryable Databricks tables for agent tools

# Convert pandas DataFrames to Spark DataFrames
spark_agent_ready = spark.createDataFrame(agent_ready_df)
spark_department_summary = spark.createDataFrame(department_summary)
spark_location_summary = spark.createDataFrame(location_summary)
spark_risk_summary = spark.createDataFrame(risk_summary)

# Save as Databricks tables
spark_agent_ready.write.mode("overwrite").saveAsTable("main.default.cleaned_onboarding_agent_ready")
spark_department_summary.write.mode("overwrite").saveAsTable("main.default.department_summary")
spark_location_summary.write.mode("overwrite").saveAsTable("main.default.location_summary")
spark_risk_summary.write.mode("overwrite").saveAsTable("main.default.risk_summary")

print("Databricks tables created:")
print("- main.default.cleaned_onboarding_agent_ready")
print("- main.default.department_summary")
print("- main.default.location_summary")
print("- main.default.risk_summary")

Databricks tables created:
- main.default.cleaned_onboarding_agent_ready
- main.default.department_summary
- main.default.location_summary
- main.default.risk_summary


The Databricks tables were created successfully in `main.default`. These tables provide queryable data sources for the agent, including the cleaned onboarding dataset, department summary table, location summary table, and risk summary table.

## 12. Verify Queryable Tables

This section confirms that the saved Databricks tables can be queried successfully. This validation step ensures that the cleaned dataset and summary tables are ready for the agent notebook.

In [0]:
# Verify Databricks tables are queryable

display(spark.sql("SELECT * FROM main.default.cleaned_onboarding_agent_ready LIMIT 5"))

display(spark.sql("""
    SELECT department, employee_count, avg_overall_onboarding_score, probation_loss_rate, high_risk_count
    FROM main.default.department_summary
    ORDER BY avg_overall_onboarding_score ASC
"""))

display(spark.sql("""
    SELECT location, employee_count, avg_overall_onboarding_score, probation_loss_rate, high_risk_count
    FROM main.default.location_summary
    ORDER BY probation_loss_rate DESC
"""))

department,location,hire_date,onboarding_survey_date,role_clarity_score,training_quality_score,manager_support_score,tooling_access_score,onboarding_nps,left_during_probation,employee_key,days_to_survey,overall_onboarding_score,low_score_count,onboarding_risk_category,date_issue_flag
Engineering,Remote-ES,2024-03-12T00:00:00.000Z,2024-04-30T00:00:00.000Z,9,4,5.0,9,6.8,0,EMP_0001,49.0,6.76,1,Medium Risk,false
Customer Success,Valencia,2025-06-01T00:00:00.000Z,2025-08-04T00:00:00.000Z,3,6,8.0,9,6.5,0,EMP_0002,64.0,6.5,1,Medium Risk,false
Engineering,Remote-ES,2024-10-25T00:00:00.000Z,2024-12-16T00:00:00.000Z,3,4,6.0,8,5.2,1,EMP_0003,52.0,5.24,2,High Risk,false
Engineering,Madrid,2025-01-04T00:00:00.000Z,2025-02-17T00:00:00.000Z,6,8,7.0,6,6.8,0,EMP_0004,44.0,6.76,0,Low Risk,false
Data,Barcelona,2025-02-01T00:00:00.000Z,2025-03-21T00:00:00.000Z,9,9,9.0,8,8.8,0,EMP_0005,48.0,8.76,0,Low Risk,false


department,employee_count,avg_overall_onboarding_score,probation_loss_rate,high_risk_count
Engineering,44,6.33,0.3,15
Finance,57,6.36,0.16,14
Product,62,6.36,0.24,18
HR,39,6.36,0.26,14
Sales,60,6.4,0.37,25
Data,49,6.41,0.14,9
Marketing,48,6.48,0.12,9
Customer Success,46,6.56,0.15,11


location,employee_count,avg_overall_onboarding_score,probation_loss_rate,high_risk_count
Seville,76,6.26,0.29,27
Valencia,74,6.51,0.24,20
Remote-ES,80,6.39,0.2,25
Madrid,95,6.49,0.2,24
Barcelona,80,6.37,0.18,19


The table checks confirm that the cleaned dataset and summary tables are available in Databricks. These outputs can now be used by the AI Engineer to build tools for employee risk analysis, department analysis, location analysis, and HR recommendation generation.

## Data Pipeline Summary

This notebook completed the Data Engineer portion of the HR Onboarding Support Agent project. The pipeline loaded the Employee Onboarding Effectiveness dataset, reviewed the dataset structure, checked missing values, validated score ranges, cleaned mixed date formats, handled missing values in `manager_support_score` and `onboarding_nps`, and created agent-ready features.

The pipeline also created an anonymized employee key, calculated days between hire date and onboarding survey date, created an overall onboarding score, counted low-score indicators, and assigned each employee an onboarding risk category.

During validation, 21 records had onboarding survey dates earlier than hire dates. These rows were kept because their onboarding scores are still useful, but a date issue flag was added and invalid `days_to_survey` values were set to missing.

The final outputs include a cleaned agent-ready dataset, department summary table, location summary table, and risk summary table. These outputs were saved as CSV files and Databricks tables so the AI Engineer can connect them to agent tools.